# 🔴 Solution: Sliding Window Attention（参考版）

## 📋 题目描述

**难度：** Medium

实现滑动窗口注意力。

滑动窗口注意力限制每个位置只关注固定窗口内的位置，在保持局部上下文的同时降低长序列的复杂度。

**签名:** `sliding_window_attention(Q, K, V, window_size) -> Tensor`

**参数:**
- `Q`, `K`, `V` — 输入张量 (B, S, D)
- `window_size` — 每个位置关注 |i-j| <= window_size 范围内的位置

**返回:** 注意力输出 (B, S, D)

**约束:**
- 用 `-inf` 掩盖 `|i - j| > window_size` 的位置
- `window_size=0` 表示每个位置只能看到自身
- 大窗口等同于全注意力

**提示：** 标准注意力 + 窗口遮蔽。构造 `|i-j|` 矩阵 → 将 `|i-j| > window_size` 的位置填 `-inf` → softmax → `@ V`。


In [ ]:
import torch
import math

In [ ]:
# ✅ SOLUTION

def sliding_window_attention(query: torch.Tensor, key: torch.Tensor, value: torch.Tensor, window_size: int, mask: torch.Tensor = None) -> torch.Tensor:
    # Sliding Window Attention
    # 每个位置只关注局部窗口内的位置，而非全部序列
    # 复杂度从 O(n²) 降到 O(n × w)，w = window_size
    # 用于 Longformer、Mistral 等长序列模型

    d_k = key.size(-1)
    seq_len = query.size(-2)

    # ---- Step 1: 计算注意力分数 ----
    scores = query @ key.transpose(-2, -1) / math.sqrt(d_k)

    # ---- Step 2: 创建滑动窗口 mask ----
    # 位置 i 只能关注 [i-window_size, i+window_size] 范围内的位置
    # i - j 的绝对值 > window_size 的位置被 mask 掉
    # torch.arange(seq_len) - torch.arange(seq_len).unsqueeze(1) 得到相对距离矩阵
    positions = torch.arange(seq_len, device=query.device)
    rel_dist = (positions.unsqueeze(0) - positions.unsqueeze(1)).abs()
    window_mask = rel_dist > window_size  # True 表示超出窗口

    # ---- Step 3: 应用 mask ----
    scores = scores.masked_fill(window_mask, float('-inf'))

    # ---- Step 4: 额外 mask（如因果 mask） ----
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    # ---- Step 5: Softmax + 加权求和 ----
    attn_weights = torch.softmax(scores, dim=-1)
    return attn_weights @ value

In [ ]:
Q=torch.randn(1,6,8); K=torch.randn(1,6,8); V=torch.randn(1,6,8)
print('window=0==V?', torch.allclose(sliding_window_attention(Q,K,V,0), V, atol=1e-5))

## 📝 核心思路总结

1. **局部注意力**：每个位置只关注窗口大小范围内的位置
2. **距离 mask**：|i-j| > window_size 的位置设为 -inf
3. **复杂度优化**：O(n²) → O(n×w)
4. **长序列处理**：Longformer、Mistral 等模型的基础